<a href="https://colab.research.google.com/github/CoolingVerseOracle/Coolingverse-data/blob/main/04_Grid_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. 파일 로드

## 1. 드라이브에서 파일 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 그리드 테이블 생성


쿨링벌스계정 드라이브내에 grid_다사_100M.shp 있음
[바로가기](https://drive.google.com/file/d/1XKjYX1erkXdhEijq6EmzxeEWPpHzjouN/view?usp=sharing)

In [ ]:
import geopandas as gpd
import pandas as pd

# 1. 100M 격자 파일 읽기
gdf = gpd.read_file('grid_다사_100M.shp')

# 2. 카카오/네이버 지도 좌표계(WGS84, EPSG:4326)로 변환
gdf_4326 = gdf.to_crs(epsg=4326)

# 3. Bounding Box (min_lng, min_lat, max_lng, max_lat) 추출
bounds = gdf_4326.bounds

# 4. grids 테이블 스키마에 맞게 컬럼 가공
grids_df = pd.DataFrame()
grids_df['grid_id'] = range(1, len(gdf_4326) + 1)
grids_df['district_id'] = 1  # 분당구 ID
grids_df['grid_code'] = gdf_4326['GRID_CD']

grids_df['min_lat'] = bounds['miny']
grids_df['min_lng'] = bounds['minx']
grids_df['max_lat'] = bounds['maxy']
grids_df['max_lng'] = bounds['maxx']

# 중심점 추출
centroids = gdf_4326.geometry.centroid
grids_df['center_lat'] = centroids.y
grids_df['center_lng'] = centroids.x

grids_df['area_km2'] = 0.01  # 100m x 100m = 0.01 km^2
grids_df['effective_area_km2'] = 0.01

# DB 적재용 CSV 변환 완료
grids_df.to_csv('grids_table_ready.csv', index=False, encoding='utf-8-sig')

/tmp/ipykernel_3053/3676194054.py:25: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = gdf_4326.geometry.centroid


[gid_table_ready 파일 바로다운받기](https://drive.google.com/file/d/1WAOeccIwdZomyghBw1EUg1MkkEoCSrof/view?usp=sharing)

## 분당구 내 그리드 필터링
단속데이터 위경도 기준 제일 낮은 데이터 소수점 둘째자리 기준 최소치는 내림, 최대치는 올림법 적용.

In [ ]:
import pandas as pd

# 1. 파일 읽기
df = pd.read_csv('grids_table_ready.csv')

# 2. 좌표 결측치(NaN) 제거
df_clean = df.dropna(subset=['center_lat', 'center_lng']).copy()

# 3. 분당구 Bounding Box 영역만 필터링 (분당구 실제 위경도 범위)
bundang_grids = df_clean[
    (df_clean['center_lat'] >= 37.27) & (df_clean['center_lat'] <= 37.50) &
    (df_clean['center_lng'] >= 126.52) & (df_clean['center_lng'] <= 127.18)
].copy()

# 4. grid_id 재정렬 (1부터 다시 부여)
bundang_grids['grid_id'] = range(1, len(bundang_grids) + 1)

# 5. 최종 파일 저장
bundang_grids.to_csv('grids_bundang_final.csv', index=False, encoding='utf-8-sig')

print(f"정제 완료! 총 {len(bundang_grids):,}개의 분당구 격자 데이터가 생성되었습니다.")

정제 완료! 총 122,318개의 분당구 격자 데이터가 생성되었습니다.


## 부천시내 그리드 필터링

In [ ]:
import pandas as pd

# 1. 파일 읽기
df = pd.read_csv('grids_table_ready.csv')

# 2. 좌표 결측치(NaN) 제거
df_clean = df.dropna(subset=['center_lat', 'center_lng']).copy()

# 3. 부천시 Bounding Box 영역만 필터링 (부천시 실제 위경도 범위)
bundang_grids = df_clean[
    (df_clean['center_lat'] >= 37.46) & (df_clean['center_lat'] <= 37.56) &
    (df_clean['center_lng'] >= 126.73) & (df_clean['center_lng'] <= 127.84)
].copy()

# 4. grid_id 재정렬 (1부터 다시 부여)
bundang_grids['grid_id'] = range(1, len(bundang_grids) + 1)

# 5. 최종 파일 저장
bundang_grids.to_csv('grids_bucheon_final.csv', index=False, encoding='utf-8-sig')

print(f"정제 완료! 총 {len(bundang_grids):,}개의 부천시 격자 데이터가 생성되었습니다.")

정제 완료! 총 55,256개의 부천시 격자 데이터가 생성되었습니다.
